<a href="https://colab.research.google.com/github/sofyadmitrieva/python-ML-basics/blob/main/hw_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Установки и импорты

In [ ]:
!pip install feedparser beautifulsoup4 -q

In [ ]:
import feedparser
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import re
from google.colab import files
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import torch
from torch.utils.data import DataLoader, TensorDataset

2. Парсинг

В качестве материала для датасета были выбраны вакансии по it-профессиям python разработчик, ML инженер и промпт инженер с сайта hh.ru

In [7]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

def get_vacancy_text(url):
    try:
        resp = requests.get(url, headers=headers, timeout=15)
        if resp.status_code != 200:
            return ""
        soup = BeautifulSoup(resp.text, "html.parser")

        # блок с описанием вакансии на hh.ru
        block = soup.find("div", {"data-qa": "vacancy-description"})
        if not block:
            return ""
        return block.get_text(separator=" ").strip()
    except Exception:
        return ""

categories = {
    "python разработчик": "python+разработчик",
    "ML инженер": "ML+инженер",
    "промпт инженер": "промпт+инженер"
}

target_total = 550
corpus = []

for label, query in categories.items():
    if len(corpus) >= target_total:
        break

    print(f"Парсим категорию: '{label}'")
    collected_per_category = 0

    for page in range(10):
        if len(corpus) >= target_total:
            break

        url = f"https://hh.ru/search/vacancy/rss?text={query}&area=1&page={page}"
        feed = feedparser.parse(url)

        if not feed.entries:
            print(f"  Страница {page} пустая, переходим к следующей категории")
            break

        for entry in feed.entries:
            if len(corpus) >= target_total:
                break

            title = entry.get("title", "").strip()
            link = entry.get("link", "")

            if not title or not link:
                continue

            # берtм полный текст со страницы вакансии
            description = get_vacancy_text(link)

            # если не удалось получить описание, то пропускаем
            if not description:
                continue

            text = f"Вакансия: {title}. Описание: {description}"
            corpus.append({"text": text, "label": label})
            collected_per_category += 1

            # пауза между запросами на страницы вакансий
            time.sleep(random.uniform(1.0, 2.0))

        print(f"  Страница {page}: итого в корпусе: {len(corpus)}")
        time.sleep(random.uniform(1.5, 2.5))

    print(f"  Всего по категории '{label}': {collected_per_category}\n")

Парсим категорию: 'python разработчик'
  Страница 0: итого в корпусе: 19
  Страница 1: итого в корпусе: 39
  Страница 2: итого в корпусе: 59
  Страница 3: итого в корпусе: 79
  Страница 4: итого в корпусе: 99
  Страница 5: итого в корпусе: 119
  Страница 6: итого в корпусе: 139
  Страница 7: итого в корпусе: 159
  Страница 8: итого в корпусе: 179
  Страница 9: итого в корпусе: 199
  Всего по категории 'python разработчик': 199

Парсим категорию: 'ML инженер'
  Страница 0: итого в корпусе: 219
  Страница 1: итого в корпусе: 239
  Страница 2: итого в корпусе: 259
  Страница 3: итого в корпусе: 279
  Страница 4: итого в корпусе: 299
  Страница 5: итого в корпусе: 319
  Страница 6: итого в корпусе: 339
  Страница 7: итого в корпусе: 359
  Страница 8: итого в корпусе: 379
  Страница 9: итого в корпусе: 399
  Всего по категории 'ML инженер': 200

Парсим категорию: 'промпт инженер'
  Страница 0: итого в корпусе: 419
  Страница 1: итого в корпусе: 439
  Страница 2: итого в корпусе: 459
  Стран

3. Выгрузка и сохранение датасета

In [8]:
df = pd.DataFrame(corpus)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Размер корпуса: {len(df)} строк")
print(f"\nРаспределение по категориям:")
print(df["label"].value_counts())
print(f"\nПример записи:")
df.head(10)

Размер корпуса: 550 строк

Распределение по категориям:
label
ML инженер            200
python разработчик    199
промпт инженер        151
Name: count, dtype: int64

Пример записи:


,text,label
0,Вакансия: Backend-разработчик Python (m/m+). О...,python разработчик
1,Вакансия: Специалист технической поддержки. Оп...,python разработчик
2,Вакансия: Бизнес ассистент / Помощник руководи...,промпт инженер
3,Вакансия: Python-разработчик. Описание: Экзон ...,python разработчик
4,Вакансия: Backend Go developer. Описание: red_...,промпт инженер
5,Вакансия: DS\ML инженер. Описание: Мы команда ...,промпт инженер
6,Вакансия: Backend Developer (Python). Описание...,python разработчик
7,Вакансия: Computer Vision Engineer. Описание: ...,ML инженер
8,Вакансия: Middle QA Engineer. Описание: Привет...,python разработчик
9,Вакансия: Python-разработчик. Описание: Обязан...,ML инженер


In [9]:
df.to_csv("it_vacancies_corpus.csv", index=False, encoding="utf-8-sig")
files.download("it_vacancies_corpus.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

4. Предобработка и разделение данных на train/val/test

In [11]:
# маппинг
label_map = {"python разработчик": 0, "ML инженер": 1, "промпт инженер": 2}
df['target'] = df['label'].map(label_map)

# разделение данных на train/validation/test
X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    df['text'].values, df['target'].values,
    test_size=0.30, random_state=42, stratify=df['target'].values
)

X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp_raw, y_temp,
    test_size=0.50, random_state=42, stratify=y_temp
)

In [12]:
df.head(10)

,text,label,target
0,Вакансия: Backend-разработчик Python (m/m+). О...,python разработчик,0
1,Вакансия: Специалист технической поддержки. Оп...,python разработчик,0
2,Вакансия: Бизнес ассистент / Помощник руководи...,промпт инженер,2
3,Вакансия: Python-разработчик. Описание: Экзон ...,python разработчик,0
4,Вакансия: Backend Go developer. Описание: red_...,промпт инженер,2
5,Вакансия: DS\ML инженер. Описание: Мы команда ...,промпт инженер,2
6,Вакансия: Backend Developer (Python). Описание...,python разработчик,0
7,Вакансия: Computer Vision Engineer. Описание: ...,ML инженер,1
8,Вакансия: Middle QA Engineer. Описание: Привет...,python разработчик,0
9,Вакансия: Python-разработчик. Описание: Обязан...,ML инженер,1


5. Создаем матрицы признаков

In [13]:
# TF-IDF
vectorizer = TfidfVectorizer(max_features=2000)
X_train = vectorizer.fit_transform(X_train_raw).toarray()
X_val = vectorizer.transform(X_val_raw).toarray()
X_test = vectorizer.transform(X_test_raw).toarray()

In [14]:
print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер валидационной выборки: {X_val.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")

Размер обучающей выборки: (385, 2000)
Размер валидационной выборки: (82, 2000)
Размер тестовой выборки: (83, 2000)


In [15]:
# смотрим одну строку
X_train[0]

array([0.        , 0.        , 0.        , ..., 0.07763588, 0.        ,
       0.        ])

In [16]:
# смотрим на срез - первые 100 признаков
X_train[0][0:100]

array([0.        , 0.        , 0.        , 0.        , 0.        ,
       0.07544997, 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.06543373, 0.        , 0.        , 0.        ,
       0.06158261, 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.12316521, 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.08838148, 0.        , 0.        , 0.12923064, 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.0843943 , 0.        , 0.        , 0.        , 0.03622

6. Перевод матриц в тензоры PyTorch

In [17]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [18]:
# как теперь выглядят наши данные внутри PyTorch
X_train_tensor[0], y_train_tensor[0]

(tensor([0.0000, 0.0000, 0.0000,  ..., 0.0776, 0.0000, 0.0000]), tensor(1))

7. Оборачиваем в датасет и создаем DataLoader

In [19]:
# оборачивает тензоры, создавая кортежи (x, y) для каждого образца данных
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [20]:
display(train_dataset)

In [21]:
# делаем DataLoaderы для деления данных на батчи по 32 штуки
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [22]:
display(train_loader)

8. Создание архитектуры

In [23]:
import torch.nn as nn
import torch.optim as optim

In [24]:
class CNN1D(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)

        self.fc1 = nn.Linear(64 * (input_dim // 4), 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.unsqueeze(1)
        # (1) свертка -> активация -> пулинг
        x = self.pool(self.relu(self.conv1(x)))
        # (2) свертка -> активация -> пулинг
        x = self.pool(self.relu(self.conv2(x)))

        # Flatten
        x = x.view(x.size(0), -1)

        # классификационные полносвязные слои
        x = self.relu(self.fc1(x))
        return self.fc2(x)

# инициализируем модель под данные
input_dim = X_train.shape[1]
num_classes = len(label_map)

model = CNN1D(input_dim, num_classes)

# функция потерь для многоклассовой классификации
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

# выводим структуру модели на экран
print(model)

CNN1D(
  (conv1): Conv1d(1, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (fc1): Linear(in_features=32000, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=3, bias=True)
  (relu): ReLU()
)


9. Обучение

In [26]:
# запускаем цикл обучения на 5 эпох
for epoch in range(5):
    total_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        # считаем ошибку
        loss = criterion(outputs, batch_y)
        # считаем градиенты
        loss.backward()
        # обновляем веса
        optimizer.step()
        total_loss += loss.item()

    # Выводим среднюю ошибку за эпоху
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1, Loss: 1.0885
Epoch 2, Loss: 1.0831
Epoch 3, Loss: 1.0272
Epoch 4, Loss: 0.9547
Epoch 5, Loss: 0.7432


10. Получаем предсказания

In [27]:
from sklearn.metrics import classification_report, accuracy_score

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        outputs = model(batch_x)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.numpy())
        all_labels.extend(batch_y.numpy())

# Вывод отчета
print("Классификационный отчет:")
print(classification_report(all_labels, all_preds, target_names=label_map.keys()))
print(f"Accuracy: {accuracy_score(all_labels, all_preds):.4f}")

Классификационный отчет:
                    precision    recall  f1-score   support

python разработчик       1.00      0.90      0.95        30
        ML инженер       0.88      0.97      0.92        30
    промпт инженер       0.96      0.96      0.96        23

          accuracy                           0.94        83
         macro avg       0.95      0.94      0.94        83
      weighted avg       0.94      0.94      0.94        83

Accuracy: 0.9398


**Анализ результатов:**

Сильные стороны: Модель отлично различает категории, что видно по высоким показателям F1-score для всех классов (от 0.92 до 0.96).

Особенности: Небольшое снижение метрик по вакансиям "ML инженер" (0.88) может быть связано с тем, что в описаниях вакансий ML-инженеров встречаются те же требования к навыкам и знаниям, что и у Python-разработчиков.

Возможное улучшение: Дальнейшее повышение качества можно достичь путем увеличения корпуса данных (сбор >1000 вакансий).